In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsx') in f:
                    file_list.append(f)
file_list

In [4]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [5]:
len(file_link)

583

In [ ]:
file_link[0]

In [7]:
att_cols=["Part Type ID","Part Type Name","Attribute Description","Required"]
df_Attributes = pd.DataFrame(columns=att_cols)
for i in range(len(file_link)):
    dfa=pd.read_excel(file_link[i],sheet_name="Attributes")
    counts=0
    for j in range(len(dfa)):
        #dfa.loc[counts, "Source"]=file_link[i]
        dfa.loc[counts, "Brand"]=file_link[i].split("Attribute_Audits\\")[1].split("_")[0]
        dfa.loc[counts, "Product_Type"]=file_link[i].split("Attribute_Audits\\")[1].split("_")[1].replace(".xlsx","")
        counts=counts+1
    df_Attributes=df_Attributes.append(dfa)

In [8]:
df_Attributes['Required']=df_Attributes['Required'].fillna('Not Mandatory')

In [ ]:
df_Attributes

In [10]:
df_Attributes['Key']=df_Attributes.Brand+df_Attributes.Product_Type+df_Attributes['Attribute Description']

In [ ]:
df_Attributes

In [12]:
len(file_link)

583

In [ ]:
con_cols=["Line","PLCD","Mfg Code","EPC Part Number","Item Number","Short Description","Brand","Product_Type"]
df_Att_values = pd.DataFrame(columns=con_cols)
for i in range(len(file_link)):
    df=pd.read_excel(file_link[i],sheet_name="Attribute Audit",skiprows=1)
    count=0
    if len(df)!=0:
        for j in range(len(df)):
            df.loc[count, "Brand"]=file_link[i].split("Attribute_Audits\\")[1].split("_")[0]
            df.loc[count, "Product_Type"]=file_link[i].split("Attribute_Audits\\")[1].split("_")[1].replace(".xlsx","")
            count=count+1
        df=df.melt(id_vars=con_cols, var_name='Attributes', value_name='Values')
        df_Att_values=df_Att_values.append(df)
    else:
        pass
df_Att_values

In [ ]:
df_Att_values.nunique()

In [15]:
df_Att_values['Key']=df_Att_values.Brand+df_Att_values.Product_Type+df_Att_values.Attributes

In [ ]:
df_Att_values

In [17]:
df_final=pd.merge(df_Att_values,df_Attributes, on='Key', how='left')

In [ ]:
df_final

In [ ]:
df_final.columns

In [20]:
df_List=df_final[['Brand_x','Product_Type_x','EPC Part Number', 'Item Number','Short Description', 'Attributes', 'Values','Part Type Name', 'Required']]

In [ ]:
df_List.shape

In [131]:
df_List=df_List[df_List["Required"]=="Mandatory"]

In [ ]:
df_List.shape

In [22]:
with pd.ExcelWriter(OFolder+'\\'+'OReilly_Mandatory_Baseline.xlsx') as writer:  # doctest: +SKIP
    df_List.to_excel(writer,index=False, sheet_name='Raw')